# Can we trust an *approximate* simulator? A bond-dimension truncation study

The previous notebook (`example_scaling_analysis.ipynb`) measured the wall: a coherent magic state
across two side-by-side distance-`d` patches needs bond dimension **χ = 32 at d=3, 256 at d=5, and an
extrapolated ≈ 2000 at d=7**. Left uncapped, the MPS grows to whatever the physics demands and the
answer is exact (to the SVD `cutoff`). That is expensive, and it is why d=7 is out of reach.

The obvious lever is to **cap the bond dimension** — set `maxdim = χ_budget` and force the MPS to
throw away everything beyond it. This makes each gate `O(χ_budget³)` regardless of what the state
*wants*, so it runs faster and larger `d` becomes reachable. The catch: the result is now an
**approximation**, and the whole point of a QEC simulation is to make quantitative claims about a
logical error rate. So the real question this notebook answers is:

> **If we deliberately under-resolve the state to go faster or reach higher `d`, *which* results can
> we still believe — and how would we know if we couldn't?**

We use the engine's `maxdim` control and its `CAP_HITS` flag (which counts gate applications where the
cap was binding) as the instruments.

## §1 — What truncation actually discards

An MPS stores, at every bond, the Schmidt (entanglement) spectrum across that cut. Both truncation
knobs act on that spectrum, differently:

- **`cutoff` (ε)** — discard Schmidt values whose *weight* is below ε. This is **error-controlled**:
  you name the error you tolerate and the bond dimension does whatever it must. A run limited only by
  `cutoff` is *exact up to ε*.
- **`maxdim` (χ_budget)** — keep at most χ_budget Schmidt values, *whatever their weight*. This is
  **cost-controlled but error-blind**: you cap the work, and the discarded weight is whatever it turns
  out to be — possibly large. `CAP_HITS` fires exactly when `maxdim`, not `cutoff`, is the binding
  constraint, i.e. when you are in this error-blind regime.

Truncation always removes the *smallest* Schmidt components. The danger for physics is that "smallest
weight" is **not** the same as "least important": the delicate quantum coherences of a logical state
often live precisely in that low-weight tail. The rest of this notebook makes that concrete.

In [ ]:
include("scaling_engine.jl");

## §2 — The convergence cliff: coherence dies before populations

Take the d=5 magic-Bell state $(\lvert00\rangle_L + e^{i\pi/4}\lvert11\rangle_L)/\sqrt2$ and sweep the
budget `maxdim`. Watch two observables: `⟨ZZ⟩ = 1` is **diagonal** (a classical correlation between
the two logical branches), while `⟨XX⟩ = 0.707` is the **off-diagonal coherence** *between* the
branches — the genuinely quantum part.

Measured (d=5, `ec=false`; exact `⟨ZZ⟩=1.000`, `⟨XX⟩=0.707`):

| maxdim | χ reached | cap_hits | ⟨ZZ⟩ | ⟨XX⟩ | time |
|---|---|---|---|---|---|
| 16  | 16  | 1227 | 1.000 | **0.000** | 54 s |
| 32  | 32  | 1163 | 1.000 | **0.000** | 113 s |
| 64  | 64  | 810  | 1.000 | **0.000** | 111 s |
| 128 | 128 | 15   | 1.000 | **0.000** | 47 s |
| 192 | 192 | 10   | 0.9995 | 0.011 | 805 s |
| 256 | 256 | 27   | 1.000 | **0.636** | 97 s |
| 512 | 256 | **0** | 1.000 | **0.707** | 41 s |

The pattern is stark and it carries three lessons:

- **`⟨ZZ⟩` is essentially immune to truncation** — the classical branch-correlation is cheap to
  represent, so it reads 1.000 even at χ=16. **`⟨XX⟩` sits at zero until the budget nears the true χ,
  then rises steeply** — a *cliff*. Below it you hold a state that is classically correct and
  quantum-mechanically hollow: the coherence that makes it a magic state has been truncated away.
- **The final χ is not the whole story — the *transient* peak is.** The `maxdim=512` row settles at
  χ=256 with `⟨XX⟩=0.707` and a clean flag; but capping *at* 256 (the `maxdim=256` row) gives only
  `⟨XX⟩=0.636` with `cap_hits=27`, because the computation transiently needs headroom *above* its
  final bond dimension and the cap clipped it. "χ reached 256, same as the converged run" looks
  reassuring and is wrong — only the non-zero flag reveals it.
- **Converged is *cheaper*, not just more correct.** The fully-resolved `maxdim=512` run is the
  *fastest* in the table (41 s), while the frustrated near-cliff `maxdim=192` run is by far the
  slowest (805 s): a maximally-truncating SVD, fighting a state it cannot represent, is expensive.
  Under-resolving to "save time" can cost both accuracy *and* time.

In [ ]:
circ = [(:H,1),(:T,1),(:CNOT,1,2)];  xx = round(ref2(ref_state(circ),:X,:X), digits=3)
println("exact ⟨ZZ⟩ = 1.0,  ⟨XX⟩ = $xx\n")
c5 = build_code(5)
for md in (64, 128, 256, 512)                     # 512 is comfortably converged; ~a few min total
    set_precision!(cutoff=1e-6, maxdim=md); Random.seed!(21)
    t = @elapsed M = run_circuit(c5, circ; ec=false, R=3, C=2, B=2, use_AD=true)
    @printf("maxdim=%-4d  χ=%-4d cap_hits=%-3d  ⟨ZZ⟩=%.3f  ⟨XX⟩=%.3f   %5.1fs\n",
            md, CHIMAX[], CAP_HITS[], corr2(M,:Z,:Z), corr2(M,:X,:X), t)
end

## §3 — `CAP_HITS` is the guardrail — but it flags *risk*, not *error size*

Look at the `cap_hits` column above. It is **non-zero exactly on the under-resolved rows** and drops
to **zero once the budget exceeds the true χ**. That makes it a reliable *alarm*: a run with
`CAP_HITS > 0` on the qubits an observable touches is, by construction, in the error-blind regime and
must not be trusted at face value.

What the flag does **not** tell you is *how wrong* you are — the `maxdim=64` and `maxdim=192` rows both
flag, but their `⟨XX⟩` errors are very different. The flag says "you are approximating here"; only a
**convergence sweep** (watching the observable plateau as you raise `maxdim`) tells you *by how much*.
So the flag is necessary, not sufficient.

## §4 — The part that should worry us: truncation biases the QEC figure of merit

`⟨XX⟩` collapsing to zero is at least *loud* — you can see it is wrong. The real hazard is a quantity
that stays plausible while being biased. The logical error rate `pL` is exactly such a quantity: it is
a single number in [0,1] with no "exact" value to eyeball against.

Below we measure `pL` for a **d=5 memory** at three `maxdim` budgets, in two bases:
- **X-memory** (`|+⟩_L`, read in X) — a logical superposition, so its representation leans on the
  entanglement tail that truncation attacks;
- **Z-memory** (`|0⟩_L`, read in Z) — comparatively classical, expected to be more robust.

`maxdim=128` is far above the memory's true χ and serves as the converged reference. Measured
(p=q=0.02, N=16, R=4):

| basis | maxdim | pL | cap_hits (total) | χ reached |
|---|---|---|---|---|
| X (`\|+⟩_L`) | 8   | **0.625** ± 0.121 | 8460 | 8 |
| X | 32  | 0.063 ± 0.061 | 3077 | 32 |
| X | 128 | 0.063 ± 0.061 | **0** | 32 |
| Z (`\|0⟩_L`) | 8   | **0.625** ± 0.121 | 8294 | 8 |
| Z | 32  | 0.063 ± 0.061 | **0** | 16 |
| Z | 128 | 0.063 ± 0.061 | **0** | 16 |

Three things to take from this, each a distinct lesson about trust:

1. **Direction of the bias.** At `maxdim=8` — below the χ the state actually needs — `pL` blows up to
   **0.625**, i.e. *worse than a coin flip*. Starving the bond dimension does not quietly flatter the
   code; it randomises the logical readout, corrupting good states into apparent failures. So here the
   bias is *pessimistic*, not falsely reassuring — but it is still a **wrong number**, and a `pL` of
   0.6 for a distance-5 memory at p=0.02 is physical nonsense.

2. **A right-looking number can still be untrustworthy.** The `X, maxdim=32` row gives `pL = 0.063` —
   *exactly* the converged answer — yet `CAP_HITS = 3077`. If you had run only that budget and reported
   0.063 at face value, you would have been **right by luck**: the flag is the only thing telling you
   the budget was binding and the agreement was not guaranteed. The `Z` memory, whose true χ is just 16,
   already reads `CAP_HITS = 0` at `maxdim=32` — genuinely converged — which is why the classical-ish
   Z memory is the more robust of the two.

3. **`pL` gives no visible warning on its own.** Unlike `⟨XX⟩`, which collapses conspicuously to zero,
   `pL = 0.063` and `pL = 0.625` are both just numbers in range. Only `CAP_HITS` and the convergence
   sweep distinguish the trustworthy row from the accidental one.

In [ ]:
nm = NoiseModel(; p=0.02, q=0.02); c5 = build_code(5)
for (sym, L) in ((:plus, :X), (:zero, :Z))
    println("-- $(L)-memory --")
    for md in (8, 32, 128)                        # 128 = converged reference; N small (cost)
        set_precision!(cutoff=1e-6, maxdim=md)
        pL, se = estimate_pL(c5, sym, L, 4, nm, 16; C=2, B=2, seed=100)
        @printf("   maxdim=%-4d  pL = %.3f ± %.3f   cap_hits(total)=%-5d χ=%d\n",
                md, pL, se, CAP_HITS[], CHIMAX[])
    end
end

## §5 — The speed / accuracy trade-off is not even monotonic

One might expect "smaller budget → faster". The `seconds` column of §2 says otherwise. Read it again:
`maxdim=192` (the frustrated, near-cliff row) is by far the **slowest** at **805 s**, while the fully
converged `maxdim=512` is the **fastest** at **41 s** — and it is also the only *correct* row for
`⟨XX⟩`. Deep truncation (16–128) is moderately quick but wrong; the danger zone is right *below* the
true χ, where every SVD maximally truncates a state it cannot represent and the arithmetic thrashes.

So the honest answer to "**can we simulate d=5 faster by capping χ?**" is nuanced:

- **For classical/diagonal observables** (syndromes, `⟨ZZ⟩`, Z-basis populations): yes — those never
  needed the full χ, so a modest cap is cheap *and* safe.
- **For coherent observables and `pL`**: only if the cap comfortably exceeds the true χ (§2's
  transient-headroom point). Cap *near or below* it and you can pay **more** time for a **wrong**
  answer — the worst of both. At d=5 the safe budget is `maxdim ≳ 512` (headroom over the true χ=256);
  anything tighter is valid only after a convergence check.

## §6 — Pushing to d=7 under a budget: where truncation quietly helps, and where it fails *loudly*

d=7 needs χ ≈ 2000 to be exact — unaffordable. Does truncation open a path anyway? The answer splits
on **how entangled the target is**, and the two halves could not be more different.

**Half 1 — cheap, low-entanglement states reach d=7 easily.** A single-patch d=7 *memory* (no
cross-patch CNOT, no teleportation) barely gets more entangled than at d=5:

| maxdim | χ reached | cap_hits | ⟨Z⟩ | time |
|---|---|---|---|---|
| 32  | 32 | 441 | 1.000 | 52 s |
| 64  | 32 | **0** | 1.000 | 33 s |
| 256 | 32 | **0** | 1.000 | 34 s |

It converges at **χ = 32**, reads the correct +1, and runs in ~30 s with a clean flag. For memory-type
and other low-entanglement / classical observables, d=7 — even d=9 — is well within reach, and a modest
cap is safe.

**Half 2 — the coherent two-qubit magic state does not, and it fails catastrophically rather than
gracefully.** The magic-Bell circuit needs χ ≈ 2000 at d=7. Capped far below that, it does **not**
return a wrong-but-finite number — the heavily-truncated state's norm collapses, the projective
measurements start sampling impossible outcomes, and the run **diverges to NaN**:

| maxdim | result |
|---|---|
| 64  | **NaN — crash** |
| 256 | **NaN — crash** |
| 384 | **NaN — crash** |

This is the sharpest trust lesson in the notebook: **at the frontier the approximation is not merely
inaccurate — it is *invalid*, and it announces that loudly**, producing NaN instead of a plausible
wrong answer. (Contrast §4's `pL`, which failed *quietly* by returning a in-range but biased number.)
A finite d=7 magic result would demand χ near the true ≈ 2000 — precisely the cost truncation was meant
to dodge. So truncation is a shortcut to d=7 *memory*, not to d=7 *non-Clifford*.

> ⚠️ The cell below runs the stable d=7 memory (~30 s), then *attempts* the d=7 magic circuit inside a
> `try/catch`. The magic attempt churns for a few minutes and then throws — that divergence is the
> intended result, not a bug.

In [ ]:
c7 = build_code(7)
# Half 1: d=7 single-patch memory — cheap, stable, correct.
set_precision!(cutoff=1e-6, maxdim=64); Random.seed!(5)
Mmem = Machine(c7); prepare_logical!(Mmem, 1, :zero); run_epoch!(Mmem, 1, 4)
@printf("d=7 memory  (maxdim=64):  χ=%d  cap_hits=%d  ⟨Z⟩=%.3f   → stable & converged\n",
        CHIMAX[], CAP_HITS[], logical_readout(Mmem, 1, :Z))

# Half 2: d=7 two-qubit magic-Bell under the same class of budget — diverges.
set_precision!(cutoff=1e-6, maxdim=256); Random.seed!(21)
try
    M2 = run_circuit(c7, [(:H,1),(:T,1),(:CNOT,1,2)]; ec=false, R=3, C=2, B=2, use_AD=true)
    @printf("d=7 magic   (maxdim=256): ⟨XX⟩=%.3f  cap_hits=%d\n", corr2(M2,:X,:X), CAP_HITS[])
catch e
    println("d=7 magic   (maxdim=256): DIVERGED — $(typeof(e)); truncation ≪ χ≈2000 collapsed the state")
end

## §7 — A trust protocol, and the verdict

Truncation converts an *exact* simulation (limited only by `cutoff`) into an *approximate* one
(limited by `maxdim`). It is a legitimate and powerful tool for **speed** and **reach**, but it moves
the burden of proof onto you. A result from a capped run is trustworthy only if:

1. **`CAP_HITS = 0`** on the observable's support — the cap never bound, so `cutoff` set the error and
   the run is exact up to ε; **or**
2. you have shown the observable **plateaus** as `maxdim` increases (a convergence sweep like §2/§4),
   so the residual truncation error is demonstrably small.

Corollaries specific to QEC, each earned by a table above:

- **Diagonal/classical observables are robust; coherent observables and `pL` are fragile.** The
  quantities a QEC study reports — logical error rates, logical coherences — are the truncation-
  sensitive ones (§2, §4). Not a coincidence: error correction is *about* the low-weight error tail,
  which is exactly what truncation discards first.
- **Truncation fails in three distinct modes — and only one is safe.** It can be *loud* (d=7 magic →
  NaN, §6: you cannot miss it), *silent* (`pL`=0.063 vs 0.625, §4: an in-range but biased number, no
  visible warning), or *silent-but-lucky* (X-memory at `maxdim=32` gave the exactly-right `pL` with
  `CAP_HITS=3077`, §4: right answer, untrustworthy provenance). The flag is what converts the two
  silent modes into something you can act on.
- **The final χ is not enough — you need headroom over the *transient* peak** (§2: `maxdim=256`
  settled at χ=256 yet lost 10% of `⟨XX⟩`; `maxdim=512` recovered it). "χ reached the converged value"
  is not the same as "the cap never bound."
- **A plausible number is not a converged number, and cheaper is not reliably faster** (§5). Never
  report a capped `pL` or coherent quantity without the sweep and a clean flag.
- **On a trusted run, `cutoff` — not `maxdim` — is the binding control.** If `maxdim` stops the bond
  growing (`CAP_HITS > 0`), you are extrapolating, not simulating.

**Verdict.** Capping bond dimension lets us reach d=7 *for low-entanglement observables* and lets us
simulate d=5 faster *when the cap clears the true χ with headroom* — genuine, useful wins. But for the
coherent, non-Clifford physics at the heart of a QEC study it buys nothing for free: the answer is
biased (silently), un-converged-but-lucky, or divergent (loudly), depending only on how far the cap
sits below what the state needs. The `CAP_HITS` flag tells us *when* we are approximating; only a
convergence sweep tells us *whether the approximation is good enough to believe*. With that discipline
the approximate simulator is trustworthy. Reported as a single capped run at face value, it is not —
and this notebook is the evidence.